# Cookie Cats: Statistical Hypothesis Testing (A/B Test)

## 1. Introduction

Following the Exploratory Data Analysis, we observed slight differences in retention rates between the **gate_30** (control) and **gate_40** (treatment) groups. However, observed differences in aggregate metrics can arise from random variation rather than a genuine behavioral shift.

The goal of this notebook is to apply rigorous statistical methods to determine whether moving the first gate from level 30 to level 40 produces a **statistically significant** impact on player retention, and, if so, whether that impact is large enough to be **practically meaningful**.

## 2. Hypotheses

We define the following hypotheses for each retention metric:

- **H₀ (Null Hypothesis):** There is no difference in retention between gate_30 and gate_40.
- **H₁ (Alternative Hypothesis):** Retention is higher when the gate is placed at level 30.

This is a **one-sided test**: we are specifically testing whether gate_30 outperforms gate_40, based on the directional signal observed in the EDA.

Significance threshold: **α = 0.05**

## 3. Methodology

We apply four complementary methods, each addressing a different dimension of the analysis:

| Method | Purpose |
|---|---|
| **Bootstrap simulation** | Estimates the sampling distribution of retention rates without parametric assumptions; calculates the empirical probability that gate_30 outperforms gate_40 |
| **Z-test for proportions** | Formal parametric test for comparing two independent binary proportions; produces a p-value and confidence interval |
| **Chi-square test** | Tests for independence between group assignment and retention outcome; validates the z-test finding via an alternative framework |
| **Cohen's h** | Quantifies the practical magnitude of the difference (effect size), independent of sample size |

## 4. Key Metrics

- **Day-1 Retention:** Whether the gate placement affects the immediate next-day return rate
- **Day-7 Retention:** Whether the gate placement impacts longer-term engagement, the metric most likely to be influenced by the gate

---

### **Setup**

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy.stats import chi2_contingency

# Set default Plotly theme
pio.templates.default = 'plotly_white'

# Consistent color palette - same as notebook 01
COLOR_MAP = {'gate_30': '#7C3AED', 'gate_40': '#F59E0B'}

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Load the cleaned dataset produced in notebook 01
df = pd.read_csv('../data/processed/cookie_cats_cleaned.csv')

print(f'Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head()

Dataset loaded: 90,188 rows, 6 columns


,userid,version,sum_gamerounds,retention_1,retention_7,segment
0,116,gate_30,3,False,False,Casual
1,337,gate_30,38,True,False,Regular
2,377,gate_40,165,True,False,Hardcore
3,483,gate_40,1,False,False,Casual
4,488,gate_40,179,True,True,Hardcore


---

### **Bootstrap Simulation**

Bootstrap resampling estimates the sampling distribution of our retention metrics empirically, without relying on parametric assumptions about the underlying data distribution.

For each of 10,000 iterations, we resample the dataset with replacement and compute the mean retention rate per group. The resulting distribution of means approximates the range of outcomes we would observe if the experiment were repeated many times under the same conditions.

We use 10,000 iterations (rather than the common 1,000) to obtain a more stable estimate of tail probabilities, particularly important for Day-7 retention, where the group difference is expected to be more pronounced

In [2]:
# Vectorized bootstrap: significantly faster than a row-level loop
# We sample indices with replacement and compute group means in each iteration
N_ITER = 10_000
n = len(df)

retention_1 = df['retention_1'].values
retention_7 = df['retention_7'].values
versions    = df['version'].values
is_gate30   = versions == 'gate_30'

boot_1d = {'gate_30': [], 'gate_40': []}
boot_7d = {'gate_30': [], 'gate_40': []}

for _ in range(N_ITER):
    idx = np.random.randint(0, n, size=n)
    mask30 = is_gate30[idx]
    boot_1d['gate_30'].append(retention_1[idx][mask30].mean())
    boot_1d['gate_40'].append(retention_1[idx][~mask30].mean())
    boot_7d['gate_30'].append(retention_7[idx][mask30].mean())
    boot_7d['gate_40'].append(retention_7[idx][~mask30].mean())

boot_1d = pd.DataFrame(boot_1d)
boot_7d = pd.DataFrame(boot_7d)

print(f'Bootstrap complete: {N_ITER:,} iterations')

Bootstrap complete: 10,000 iterations


### Day-1 Retention

In [3]:
# Overlay KDE curves for the bootstrapped Day-1 retention distributions
fig = go.Figure()

for version, color in COLOR_MAP.items():
    fig.add_trace(go.Violin(
        x=boot_1d[version],
        name=version,
        line_color=color,
        fillcolor=color,
        opacity=0.5,
        side='positive',
        orientation='h',
        showlegend=True
    ))

fig.update_layout(
    title='Bootstrapped Distribution of Day-1 Retention Means',
    xaxis_title='Retention Rate',
    yaxis_title='Group',
    violingap=0.4
)
fig.show()

In [4]:
# Compute the percentage lift of gate_30 over gate_40 in each bootstrap iteration
boot_1d['diff'] = (boot_1d['gate_30'] - boot_1d['gate_40']) / boot_1d['gate_40'] * 100

prob_1d = (boot_1d['diff'] > 0).mean()
ci_low_1d, ci_high_1d = boot_1d['diff'].quantile([0.025, 0.975])

print(f'Probability that Day-1 retention is higher for gate_30: {prob_1d*100:.2f}%')
print(f'95% CI of the difference: [{ci_low_1d:.3f}%, {ci_high_1d:.3f}%]')

Probability that Day-1 retention is higher for gate_30: 96.75%
95% CI of the difference: [-0.084%, 2.813%]


In [5]:
# Distribution of the percentage difference: values above 0 favour gate_30
fig = px.histogram(
    boot_1d,
    x='diff',
    nbins=80,
    color_discrete_sequence=['#7C3AED'],
    labels={'diff': '% Difference (gate_30 - gate_40)', 'count': 'Frequency'},
    title='Bootstrap Distribution of Day-1 Retention Lift (gate_30 vs gate_40)'
)
fig.add_vline(x=0, line_dash='dash', line_color='#E11D48',
              annotation_text='H₀: no difference', annotation_position='top left')
fig.add_vline(x=ci_low_1d, line_dash='dot', line_color='gray',
              annotation_text='2.5th pct', annotation_position='bottom right')
fig.add_vline(x=ci_high_1d, line_dash='dot', line_color='gray',
              annotation_text='97.5th pct', annotation_position='bottom left')
fig.show()

The bootstrapped distribution of Day-1 retention lift is centered slightly above zero, but the 95% confidence interval spans both sides of the null. This indicates that the Day-1 difference between groups is weak and likely not statistically significant, a finding we will confirm with formal tests below

### Day-7 Retention

In [6]:
# Bootstrapped Day-7 retention distributions per group
fig = go.Figure()

for version, color in COLOR_MAP.items():
    fig.add_trace(go.Violin(
        x=boot_7d[version],
        name=version,
        line_color=color,
        fillcolor=color,
        opacity=0.5,
        side='positive',
        orientation='h',
        showlegend=True
    ))

fig.update_layout(
    title='Bootstrapped Distribution of Day-7 Retention Means',
    xaxis_title='Retention Rate',
    yaxis_title='Group',
    violingap=0.4
)
fig.show()

In [7]:
# Percentage lift for Day-7 and 95% bootstrap confidence interval
boot_7d['diff'] = (boot_7d['gate_30'] - boot_7d['gate_40']) / boot_7d['gate_40'] * 100

prob_7d = (boot_7d['diff'] > 0).mean()
ci_low_7d, ci_high_7d = boot_7d['diff'].quantile([0.025, 0.975])

print(f'Probability that Day-7 retention is higher for gate_30: {prob_7d*100:.2f}%')
print(f'95% CI of the difference: [{ci_low_7d:.3f}%, {ci_high_7d:.3f}%]')

Probability that Day-7 retention is higher for gate_30: 99.94%
95% CI of the difference: [1.708%, 7.349%]


In [8]:
# Distribution of Day-7 lift: the separation from zero is expected to be much clearer than Day-1
fig = px.histogram(
    boot_7d,
    x='diff',
    nbins=80,
    color_discrete_sequence=['#7C3AED'],
    labels={'diff': '% Difference (gate_30 - gate_40)', 'count': 'Frequency'},
    title='Bootstrap Distribution of Day-7 Retention Lift (gate_30 vs gate_40)'
)
fig.add_vline(x=0, line_dash='dash', line_color='#E11D48',
              annotation_text='H₀: no difference', annotation_position='top left')
fig.add_vline(x=ci_low_7d, line_dash='dot', line_color='gray',
              annotation_text='2.5th pct', annotation_position='bottom right')
fig.add_vline(x=ci_high_7d, line_dash='dot', line_color='gray',
              annotation_text='97.5th pct', annotation_position='bottom left')
fig.show()

The Day-7 lift distribution is substantially shifted to the right of zero, with the 95% confidence interval entirely above the null. This is a strong signal that gate_30 produces meaningfully higher 7-day retention than gate_40

---

### **Z-Test for Proportions**

We now apply a formal one-sided z-test for proportions to each retention metric. This test assumes that the sample proportions follow an approximately normal distribution, a reasonable assumption given our large sample size (n > 40,000 per group).

We also compute a 95% confidence interval for each group's retention rate to quantify estimation uncertainty

In [9]:
def run_ztest(df, metric):
    """Run a one-sided z-test for proportions and return a results summary."""
    successes = df.groupby('version')[metric].sum()
    n_obs     = df.groupby('version')[metric].count()

    # Order: [gate_30 (control), gate_40 (treatment)]
    counts = [successes['gate_30'], successes['gate_40']]
    nobs   = [n_obs['gate_30'],     n_obs['gate_40']]

    z_stat, p_value = proportions_ztest(count=counts, nobs=nobs, alternative='larger')

    ci_30 = proportion_confint(counts[0], nobs[0], alpha=0.05, method='wilson')
    ci_40 = proportion_confint(counts[1], nobs[1], alpha=0.05, method='wilson')

    rate_30 = counts[0] / nobs[0]
    rate_40 = counts[1] / nobs[1]

    return {
        'metric':         metric,
        'rate_gate30':    round(rate_30 * 100, 4),
        'ci_gate30':      (round(ci_30[0]*100, 4), round(ci_30[1]*100, 4)),
        'rate_gate40':    round(rate_40 * 100, 4),
        'ci_gate40':      (round(ci_40[0]*100, 4), round(ci_40[1]*100, 4)),
        'z_statistic':    round(z_stat, 4),
        'p_value':        round(p_value, 6),
        'significant':    p_value < 0.05
    }

result_1d = run_ztest(df, 'retention_1')
result_7d = run_ztest(df, 'retention_7')

for r in [result_1d, result_7d]:
    print(f"=== {r['metric']} ===")
    print(f"  gate_30: {r['rate_gate30']:.4f}%  95% CI: {r['ci_gate30']}")
    print(f"  gate_40: {r['rate_gate40']:.4f}%  95% CI: {r['ci_gate40']}")
    print(f"  Z-statistic: {r['z_statistic']}  |  p-value: {r['p_value']}")
    print(f"  Reject H₀: {r['significant']}")
    print()

=== retention_1 ===
  gate_30: 44.8198%  95% CI: (44.3592, 45.2812)
  gate_40: 44.2283%  95% CI: (43.7724, 44.6851)
  Z-statistic: 1.7871  |  p-value: 0.03696
  Reject H₀: True

=== retention_7 ===
  gate_30: 19.0183%  95% CI: (18.6572, 19.3848)
  gate_40: 18.2000%  95% CI: (17.8481, 18.5573)
  Z-statistic: 3.1574  |  p-value: 0.000796
  Reject H₀: True



In [13]:
# Side-by-side subplots: one panel per retention metric, groups clearly separated on x-axis
# This layout makes it easier to assess CI overlap within each metric independently
ci_data = []
for r in [result_1d, result_7d]:
    metric_label = 'Day-1' if '1' in r['metric'] else 'Day-7'
    for version, rate, ci in [
        ('gate_30', r['rate_gate30'], r['ci_gate30']),
        ('gate_40', r['rate_gate40'], r['ci_gate40'])
    ]:
        ci_data.append({
            'metric': metric_label, 'version': version,
            'rate': rate, 'ci_low': ci[0], 'ci_high': ci[1]
        })

ci_df = pd.DataFrame(ci_data)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Day-1 Retention', 'Day-7 Retention'],
    shared_yaxes=False
)

for col, metric in enumerate(['Day-1', 'Day-7'], start=1):
    subset = ci_df[ci_df['metric'] == metric]
    for _, row in subset.iterrows():
        fig.add_trace(
            go.Scatter(
                x=[row['version']],
                y=[row['rate']],
                error_y=dict(
                    type='data',
                    array=[row['ci_high'] - row['rate']],
                    arrayminus=[row['rate'] - row['ci_low']],
                    visible=True,
                    thickness=2,
                    width=8
                ),
                mode='markers',
                marker=dict(color=COLOR_MAP[row['version']], size=12),
                name=row['version'],
                showlegend=(col == 1)
            ),
            row=1, col=col
        )

fig.update_layout(
    title_text='Retention Rates with 95% Confidence Intervals by Metric',
    yaxis_title='Retention Rate (%)',
    yaxis2_title='Retention Rate (%)'
)
fig.show()


The confidence interval chart confirms the z-test results: for Day-1 retention, the confidence intervals partially overlap and the p-value is close to the significance threshold, indicating a weak (though statistically detectable) difference. For Day-7 retention, the confidence intervals are clearly separated, corroborating the bootstrap findings and providing strong evidence that gate_30 produces meaningfully higher long-term retention

### **Chi-Square Test of Independence**

The chi-square test of independence assesses whether group assignment (gate_30 vs gate_40) and retention outcome (retained vs churned) are statistically independent. It provides an alternative framework for validating the z-test results without relying on the normal approximation

In [11]:
def run_chisquare(df, metric):
    """Build a contingency table and run a chi-square test of independence."""
    contingency = pd.crosstab(df['version'], df[metric])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    return {
        'metric':      metric,
        'chi2':        round(chi2, 4),
        'p_value':     round(p_value, 6),
        'dof':         dof,
        'significant': p_value < 0.05,
        'contingency': contingency
    }

chi_1d = run_chisquare(df, 'retention_1')
chi_7d = run_chisquare(df, 'retention_7')

for r in [chi_1d, chi_7d]:
    print(f"=== {r['metric']} ===")
    print(r['contingency'])
    print(f"  Chi-square statistic: {r['chi2']}  |  p-value: {r['p_value']}  |  df: {r['dof']}")
    print(f"  Reject H₀: {r['significant']}")
    print()

=== retention_1 ===
retention_1  False  True 
version                  
gate_30      24665  20034
gate_40      25370  20119
  Chi-square statistic: 3.1698  |  p-value: 0.07501  |  df: 1
  Reject H₀: False

=== retention_7 ===
retention_7  False  True 
version                  
gate_30      36198   8501
gate_40      37210   8279
  Chi-square statistic: 9.9153  |  p-value: 0.001639  |  df: 1
  Reject H₀: True



The chi-square results are consistent with the z-test: no significant association between group and Day-1 retention, but a significant association for Day-7 retention. The convergence of two independent testing frameworks strengthens confidence in the finding

---

### **Effect Size - Cohen's h**

Statistical significance alone does not convey the practical magnitude of an effect. With a large sample (n ≈ 90,000), even trivially small differences can produce significant p-values.

**Cohen's h** is the standard effect size measure for comparing two proportions. It is calculated as:

$$h = 2 \arcsin(\sqrt{p_1}) - 2 \arcsin(\sqrt{p_2})$$

Interpretation guidelines (Cohen, 1988):

| \|h\| | Interpretation |
|---|---|
| < 0.20 | Small effect |
| 0.20 – 0.50 | Medium effect |
| > 0.50 | Large effect |

In [12]:
def cohens_h(p1, p2):
    """Compute Cohen's h effect size for two proportions."""
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

def interpret_cohens_h(h):
    """Return a qualitative interpretation of Cohen's h."""
    h = abs(h)
    if h < 0.20:
        return 'Small'
    elif h < 0.50:
        return 'Medium'
    else:
        return 'Large'

for r in [result_1d, result_7d]:
    p30 = r['rate_gate30'] / 100
    p40 = r['rate_gate40'] / 100
    h   = cohens_h(p30, p40)
    print(f"{r['metric']}:  Cohen's h = {h:.4f}  ({interpret_cohens_h(h)} effect)")

retention_1:  Cohen's h = 0.0119  (Small effect)
retention_7:  Cohen's h = 0.0210  (Small effect)


Both effect sizes fall in the **small** range, which is typical for A/B tests in consumer apps at scale. A small Cohen's h does not diminish the practical relevance of the finding, at 90,000 players, even a small retention improvement translates to a meaningful number of retained users and a measurable revenue impact

---

### **Results Summary**

The table below consolidates all findings across methods and metrics

In [14]:
# Consolidated results table
summary = pd.DataFrame([
    {
        'Metric':            'Day-1 Retention',
        'gate_30 rate (%)':  result_1d['rate_gate30'],
        'gate_40 rate (%)':  result_1d['rate_gate40'],
        'Bootstrap P(30>40)': f"{prob_1d*100:.1f}%",
        'Z-test p-value':    result_1d['p_value'],
        'Chi-sq p-value':    chi_1d['p_value'],
        "Cohen's h":         round(cohens_h(result_1d['rate_gate30']/100, result_1d['rate_gate40']/100), 4),
        'Significant':       result_1d['significant']
    },
    {
        'Metric':            'Day-7 Retention',
        'gate_30 rate (%)':  result_7d['rate_gate30'],
        'gate_40 rate (%)':  result_7d['rate_gate40'],
        'Bootstrap P(30>40)': f"{prob_7d*100:.1f}%",
        'Z-test p-value':    result_7d['p_value'],
        'Chi-sq p-value':    chi_7d['p_value'],
        "Cohen's h":         round(cohens_h(result_7d['rate_gate30']/100, result_7d['rate_gate40']/100), 4),
        'Significant':       result_7d['significant']
    }
])

summary

,Metric,gate_30 rate (%),gate_40 rate (%),Bootstrap P(30>40),Z-test p-value,Chi-sq p-value,Cohen's h,Significant
0,Day-1 Retention,44.8198,44.2283,96.8%,0.036960,0.075010,0.0119,True
1,Day-7 Retention,19.0183,18.2000,99.9%,0.000796,0.001639,0.0210,True


### **Conclusion and Business Recommendation**

### Statistical Findings

| Metric | Finding |
|---|---|
| Day-1 Retention | Statistically significant difference detected (p < 0.05), but with a very small effect size and CI overlap - the practical impact is negligible |
| Day-7 Retention | Statistically significant advantage for gate_30 (p < 0.05 across all tests, bootstrap P > 99%) |
| Effect size | Small (Cohen's h < 0.20), consistent with typical A/B test magnitudes at scale |

### Interpretation

The gate position produces a statistically significant (p < 0.05) but practically negligible difference in Day-1 retention, the effect size is very small and the confidence intervals nearly overlap, suggesting this finding may be an artifact of the large sample size rather than a meaningful behavioral difference. Most players have not yet encountered either gate within their first session, which likely explains the absence of a strong Day-1 effect.

Over a 7-day window, however, the earlier gate (level 30) produces both a statistically significant and more practically meaningful retention advantage. This is the metric most directly relevant to the gate position experiment.

This is consistent with the **hedonic adaptation** hypothesis: players who are forced to pause earlier (at level 30) experience a brief break that refreshes their engagement with the game, increasing the probability of a return visit over the following week.

### Business Recommendation

> **Keep the gate at level 30.**

The evidence across three independent statistical methods consistently supports gate_30 for Day-7 retention. Although the effect size is small, the scale of the player base means that even a fractional retention improvement translates to a significant number of additional returning players, and corresponding revenue impact from in-app purchases.

**Caveats and next steps:**
- This dataset does not include monetization data; a complete recommendation should incorporate revenue-per-user metrics alongside retention.
- The analysis treats all players uniformly. A segment-level recommendation (e.g., applying different gates to casual vs. hardcore players) could yield stronger outcomes.
- A Bayesian analysis (notebook 03) will complement these findings by quantifying the posterior probability that gate_30 is the optimal configuration.

**Next step:** `03_bayesian.ipynb` - Bayesian A/B testing to quantify the posterior probability that gate_30 is the better configuration